# Task A Run 11 -- one reinitialized layer, full data

The current best Task A submission with **exactly one flag changed**.

Run 9 scored `0.8187` macro-F1 and `0.8189` accuracy: a fixed blend of a full-fit
TF-IDF/SVM and a full-data MuRIL. Its MuRIL component reinitializes the **top two** encoder
layers, a default inherited rather than chosen. Task B measured one layer against none at
**+3.0 points** and one against two at +0.5. Task A has never tested either.

| setting | Run 9, the current best | this run |
|---|---|---|
| SVM | demojized, `--full-fit`, all 6,401 rows | same |
| MuRIL | demojized, `--folds 1`, 1 seed, 6 epochs, effective batch 16 | same |
| **reinitialized layers** | **2** | **1** |
| blend weights | 0.57 SVM / 0.43 MuRIL | same |
| decision threshold | 0.5 | same |

Everything else is held fixed, including the blend weights. Those were fitted in Run 8
against a two-layer MuRIL, so they are arguably no longer optimal, but refitting them would
be a second change and there is no holdout here to refit them on honestly. One change at a
time is what makes a CodaBench gap attributable.

## If it wins, it becomes the base

A better MuRIL component is a better foundation for everything queued after it: TAPT,
the external corpus's labels, more seeds, more epochs. Those all stack on whichever
reinitialization setting wins here.

## No local score, by construction

Every labelled row is in training, so nothing is held out and the official validation
labels are never released. CodaBench is the only readout. Two label-free diagnostics are
printed instead, and neither is a score: the predicted class balance against the training
prior, which catches a collapse, and the agreement with Run 9's predictions, which says
whether this is a near-rerun or a genuinely different bet.

## Runtime

About **40 minutes**: two for the SVM, roughly 33 for a single MuRIL seed on 6,401 rows.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Train both components on all labelled data

`--full-fit` skips the SVM's OOF pass. `--folds 1` is a true full-data MuRIL fit with no
15% holdout. Both write probabilities for the official validation inputs.

In [ ]:
SVM_TAG, MURIL_TAG = "task_a_r11_svm", "task_a_r11_muril"

run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_TAG, "--demojize", "--full-fit"],
    log=f"artifacts/logs/{SVM_TAG}.log")

run([sys.executable, "-u", "-m", "hastika.models.muril",
     "--tag", MURIL_TAG, "--model", "google/muril-base-cased",
     "--folds", "1", "--seeds", "42", "--epochs", "6",
     "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
     "--select", "last", "--reinit-layers", "1"],
    log=f"artifacts/logs/{MURIL_TAG}.log")

import re
log = pathlib.Path(f"artifacts/logs/{MURIL_TAG}.log").read_text()
fits = re.findall(r"seed (\d+) FULL FIT, (\d+) rows", log)
print("full fits:", fits)
assert fits and all(int(n) == 6401 for _, n in fits), "a seed did not see all 6,401 rows"
assert "reinit=1" in log, "training log does not show one-layer reinitialization"

## 2. Apply Run 9's blend unchanged

0.57 SVM and 0.43 MuRIL, threshold 0.5. Carried over deliberately: refitting them without
a holdout would mean fitting on the hidden labels, and changing them would confound the
result with the reinitialization change this run exists to measure.

In [ ]:
W_SVM, THRESH = 0.57, 0.5
svm_p = np.load(pathlib.Path("artifacts/runs") / SVM_TAG / "test_probs.npy")
muril_p = np.load(pathlib.Path("artifacts/runs") / MURIL_TAG / "test_probs.npy")
assert svm_p.shape == muril_p.shape, (svm_p.shape, muril_p.shape)

ids = pd.read_csv("data/raw/binary_validation_inputs.csv")
p = W_SVM * svm_p[:, 1] + (1 - W_SVM) * muril_p[:, 1]
labels = np.where(p > THRESH, "Hate", "Non-Hate")

out = pathlib.Path("artifacts/runs") / "task_a_r11_blend"
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": ids["id"], "label": labels}).to_csv(out / "predictions.csv", index=False)
print(pd.Series(labels).value_counts().to_dict())

## 3. Label-free diagnostics

Neither of these is a score. The first catches a collapsed model, which is the only failure
detectable without labels. The second says how far this run sits from the submission it is
trying to beat: near-identical predictions mean a near-identical score, whatever the
reinitialization did to the encoder.

In [ ]:
train_rate = (train.iloc[keep]["Label"] == "Hate").mean()
print(f"predicted Hate rate {(labels == 'Hate').mean():.3f}   training prior {train_rate:.3f}")

prev = pathlib.Path("submissions/task_a/task_a_predictions.zip")
if prev.exists():
    with zipfile.ZipFile(prev) as z:
        old = pd.read_csv(z.open([n for n in z.namelist() if n.endswith(".csv")][0]))
    m = pd.DataFrame({"id": ids["id"], "label": labels}).merge(old, on="id",
                                                              suffixes=("_new", "_old"))
    print(f"agreement with the preserved Task A submission: "
          f"{(m['label_new'] == m['label_old']).mean():.3f} over {len(m)} rows")
print("\nmajority-class baseline is 0.491; a Hate rate outside 0.35-0.65 means check the log")

## 4. Package the submission

In [ ]:
ZIP = "/kaggle/working/task_a_reinit1_blend.zip"
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "a", "--pred", str(out / "predictions.csv"), "--out", ZIP])
with zipfile.ZipFile(ZIP) as f:
    assert f.namelist() == ["predictions.csv"], f.namelist()
print("ready to upload:", ZIP, pathlib.Path(ZIP).stat().st_size, "bytes")

## 5. Preserve outputs

Keep both `test_probs.npy` files. They are what lets a future blend weight or threshold be
tried without retraining either component.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_reinit1_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / pathlib.Path(ZIP).name)
shutil.copy2(out / "predictions.csv", OUT / "predictions.csv")
for tag in [SVM_TAG, MURIL_TAG]:
    shutil.copy2(pathlib.Path("artifacts/runs") / tag / "test_probs.npy",
                 OUT / f"{tag}_test_probs.npy")
    shutil.copy2(f"artifacts/logs/{tag}.log", OUT / f"{tag}.log")
json.dump({"w_svm": W_SVM, "threshold": THRESH, "reinit_layers": 1, "seeds": ["42"]},
          open(OUT / "config.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 6. After CodaBench scores it

Record the score in `docs/EXPERIMENTS.md` and `submissions/README.md` against Run 9's
`0.8187`, and preserve the ZIP under `submissions/` the way the other entries are.

**If it beats 0.8187, one layer becomes the base recipe** and every queued experiment
stacks on it: Run 12's TAPT, Run 13's external labels, more seeds, more epochs.

**If it loses by more than about 3 points**, two layers is the better setting for Task A
and the queued experiments should keep `--reinit-layers 2`.

**If the gap is under 3 points, it is not resolved.** One score on 806 rows carries roughly
1.5 points of standard deviation. In that case prefer one layer anyway, on the grounds that
Task B measured reinitialization mattering and this run shows the count does not, but say
so rather than claiming a result.